In [8]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset


# 1. Settings

torch.manual_seed(42)
np.random.seed(42)

CSV_FILE = "modelling.csv"

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 0.001

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# 2. Choose the prediction criteria

# OPTION 1:
# Use every suitable column automatically
FEATURE_COLUMNS = ["StoreType"]


# OPTION 2:
# Comment out FEATURE_COLUMNS = None above and use
# your own selection instead.
#
# FEATURE_COLUMNS = [
#     "StoreType",
#     "Assortment",
#     "DayOfWeek",
#     "Promo",
#     "CompetitionDistance",
#     "SchoolHoliday"
# ]


# Columns that should never be used as inputs
EXCLUDED_COLUMNS = [
    "Sales",      # This is what we are predicting
    "Customers",  # Usually unknown before sales occur
    "Date"        # Year, Month, Day already represent the date
]


# 3. Import the CSV

df = pd.read_csv(
    CSV_FILE,
    low_memory=False
)

print("Dataset size:", df.shape)


# 4. Select input columns and target

if FEATURE_COLUMNS is None:

    feature_columns = [
        column
        for column in df.columns
        if column not in EXCLUDED_COLUMNS
    ]

else:

    feature_columns = FEATURE_COLUMNS.copy()

    invalid_columns = [
        column
        for column in feature_columns
        if column not in df.columns
    ]

    if invalid_columns:
        raise ValueError(
            f"Columns not found: {invalid_columns}"
        )

    forbidden_columns = [
        column
        for column in feature_columns
        if column in EXCLUDED_COLUMNS
    ]

    if forbidden_columns:
        raise ValueError(
            f"These columns cannot be inputs: "
            f"{forbidden_columns}"
        )


X = df[feature_columns].copy()
y = df[["Sales"]].copy()

print("\nColumns being used:")
for column in feature_columns:
    print("-", column)


# 5. Find numerical and categorical columns

numerical_columns = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("\nNumerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)


# 6. Split the dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


# 7. Prepare numerical columns

numerical_pipeline = Pipeline([
    (
        "fill_missing",
        SimpleImputer(strategy="median")
    ),
    (
        "scale",
        StandardScaler()
    )
])


# 8. Prepare categorical columns

categorical_pipeline = Pipeline([
    (
        "fill_missing",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encode",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


# 9. Combine preprocessing

transformers = []

if numerical_columns:
    transformers.append(
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        )
    )

if categorical_columns:
    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "\nNumber of inputs after preprocessing:",
    X_train_processed.shape[1]
)


# 10. Scale the Sales target

sales_scaler = StandardScaler()

y_train_scaled = sales_scaler.fit_transform(
    y_train
)

y_test_scaled = sales_scaler.transform(
    y_test
)


# 11. Convert into PyTorch tensors

X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_scaled,
    dtype=torch.float32
)


# 12. Create DataLoaders

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# 13. Define the neural network

class SalesNeuralNetwork(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


model = SalesNeuralNetwork(
    input_size=X_train_tensor.shape[1]
).to(device)

print("\nNeural network:")
print(model)


# 14. Loss and optimizer

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# 15. Train the model

best_test_loss = float("inf")
best_model_weights = None

for epoch in range(EPOCHS):

    # Training
    model.train()

    total_training_loss = 0

    for inputs, targets in train_loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        predictions = model(inputs)

        loss = criterion(
            predictions,
            targets
        )

        loss.backward()

        optimizer.step()

        total_training_loss += (
            loss.item() * inputs.size(0)
        )

    training_loss = (
        total_training_loss / len(train_dataset)
    )

    # Testing
    model.eval()

    total_test_loss = 0

    with torch.no_grad():

        for inputs, targets in test_loader:

            inputs = inputs.to(device)
            targets = targets.to(device)

            predictions = model(inputs)

            loss = criterion(
                predictions,
                targets
            )

            total_test_loss += (
                loss.item() * inputs.size(0)
            )

    test_loss = (
        total_test_loss / len(test_dataset)
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Training loss: {training_loss:.4f} | "
        f"Testing loss: {test_loss:.4f}"
    )

    if test_loss < best_test_loss:

        best_test_loss = test_loss

        best_model_weights = copy.deepcopy(
            model.state_dict()
        )


# Restore the best weights
model.load_state_dict(best_model_weights)


# 16. Make sales predictions

model.eval()

scaled_predictions = []

with torch.no_grad():

    for inputs, _ in test_loader:

        inputs = inputs.to(device)

        predictions = model(inputs)

        scaled_predictions.append(
            predictions.cpu().numpy()
        )

scaled_predictions = np.vstack(
    scaled_predictions
)


# Convert back into real sales values
predicted_sales = sales_scaler.inverse_transform(
    scaled_predictions
).flatten()

predicted_sales = np.maximum(
    predicted_sales,
    0
)

actual_sales = y_test["Sales"].to_numpy()


# 17. Evaluate accuracy

mae = mean_absolute_error(
    actual_sales,
    predicted_sales
)

rmse = np.sqrt(
    np.mean(
        (actual_sales - predicted_sales) ** 2
    )
)

nonzero = actual_sales != 0

mape = np.mean(
    np.abs(
        (
            actual_sales[nonzero]
            - predicted_sales[nonzero]
        )
        / actual_sales[nonzero]
    )
) * 100

print("\nFinal results")
print(f"Mean Absolute Error:       {mae:.2f}")
print(f"Root Mean Squared Error:   {rmse:.2f}")
print(f"Mean Absolute Percentage:  {mape:.2f}%")


# 18. Display and save predictions

results = X_test.reset_index(drop=True).copy()

results["ActualSales"] = actual_sales

results["PredictedSales"] = (
    predicted_sales.round().astype(int)
)

results["AbsoluteError"] = abs(
    results["ActualSales"]
    - results["PredictedSales"]
)

print("\nExample predictions:")
print(results.head(20))

results.to_csv(
    "sales_predictions.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "feature_columns": feature_columns,
        "model_input_size": X_train_tensor.shape[1],
        "sales_mean": sales_scaler.mean_,
        "sales_scale": sales_scaler.scale_
    },
    "sales_model.pth"
)

print("\nSaved sales_predictions.csv")
print("Saved sales_model.pth")

Using device: cpu
Dataset size: (175000, 22)

Columns being used:
- StoreType

Numerical columns: ['StoreType']
Categorical columns: []

Training rows: 140000
Testing rows: 35000

Number of inputs after preprocessing: 1

Neural network:
SalesNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=1, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=16, bias=True)
    (6): ReLU()
    (7): Linear(in_features=16, out_features=1, bias=True)
  )
)
Epoch 01/50 | Training loss: 0.9892 | Testing loss: 0.9879
Epoch 02/50 | Training loss: 0.9801 | Testing loss: 0.9848
Epoch 03/50 | Training loss: 0.9793 | Testing loss: 0.9840
Epoch 04/50 | Training loss: 0.9787 | Testing loss: 0.9845
Epoch 05/50 | Training loss: 0.9786 | Testing loss: 0.9846
Epoch 06/50 | Training loss: 0.9786 | Testing loss: 0.9846
Epoch 07/50 | Training loss: 0.97

SyntaxError: invalid syntax (1986530368.py, line 1)

SyntaxError: invalid syntax (952820021.py, line 1)